In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import scipy.stats as stats
from tqdm import tqdm

In [2]:
# --- GENERAL VISUAL CONFIGURATION ---
mpl.rcParams['hatch.linewidth'] = 0.4  
mpl.rcParams['hatch.color'] = 'black'

AXIS_FONT_SIZE = 12
CBAR_FONT_SIZE = 12
TITLE_FONT_SIZE = 14

# --- PATH CONFIGURATION ---
base_path = r""
file_sm = base_path + "Soil_moisture_monthly_COMBINED_1980_2022_box_NEW.nc"
file_era5 = base_path + "volumetric_ERA5_land_1980_2022_box_interpolated_NEW.nc"
file_mask = base_path + "mascara_final_egipto_v2.nc"


print(f"Soil moisture file: {file_sm}")
print(f"ERA5 file: {file_era5}")
print(f"Mask file: {file_mask}")

Soil moisture file: Soil_moisture_monthly_COMBINED_1980_2022_box_NEW.nc
ERA5 file: volumetric_ERA5_land_1980_2022_box_interpolated_NEW.nc
Mask file: mascara_final_egipto_v2.nc


In [3]:
# --- DATA LOADING ---
print("\n[INFO] Loading datasets...")
ds_sm = xr.open_dataset(file_sm)
ds_era = xr.open_dataset(file_era5)
ds_mask = xr.open_dataset(file_mask)

print("[INFO] Datasets loaded successfully")
print(f"SM dimensions: {ds_sm.dims}")
print(f"ERA5 dimensions: {ds_era.dims}")
print(f"Mask dimensions: {ds_mask.dims}")

# --- ALIGNMENT ---
print("\n[INFO] Aligning datasets...")
ds_sm, ds_era = xr.align(ds_sm, ds_era, join="override")
print("[INFO] Alignment completed")

layers = {
    "Layer 1": ds_era["swvl1"],
    "Layer 2": ds_era["swvl1"] + ds_era["swvl2"],
    "Layer 3": ds_era["swvl1"] + ds_era["swvl2"] + ds_era["swvl3"]
}



[INFO] Loading datasets...
[INFO] Datasets loaded successfully
SM dimensions: FrozenMappingWarningOnValuesAccess({'time': 516, 'bnds': 2, 'lon': 252, 'lat': 64})
ERA5 dimensions: FrozenMappingWarningOnValuesAccess({'valid_time': 516, 'lon': 252, 'lat': 64})
Mask dimensions: FrozenMappingWarningOnValuesAccess({'lat': 171, 'lon': 611})

[INFO] Aligning datasets...
[INFO] Alignment completed


In [6]:
def get_corr_map_memory_safe(da_sm, da_era, layer_name):
    """Compute correlation in a memory-safe way using NumPy and show progress."""
    
    print(f"\n[INFO] Starting correlation for {layer_name}")
    
    sm_vals = da_sm.values
    era_vals = da_era.values
    
    n_lat, n_lon = sm_vals.shape[1], sm_vals.shape[2]
    
    print(f"[INFO] Grid size: {n_lat} lat x {n_lon} lon")
    print(f"[INFO] Time steps: {sm_vals.shape[0]}")

In [ ]:

def get_corr_map_memory_safe(da_sm, da_era, layer_name):
    """Compute correlation in a memory-safe way using NumPy and show progress."""
    
    print(f"\n[INFO] Starting correlation for {layer_name}")
    
    sm_vals = da_sm.values
    era_vals = da_era.values
    
    n_lat, n_lon = sm_vals.shape[1], sm_vals.shape[2]
    
    print(f"[INFO] Grid size: {n_lat} lat x {n_lon} lon")
    print(f"[INFO] Time steps: {sm_vals.shape[0]}")
    
    corr = np.full((n_lat, n_lon), np.nan)
    p_values = np.full((n_lat, n_lon), np.nan)
    
    for i in tqdm(range(n_lat), desc=f"{layer_name}", unit="lat"):
        x = sm_vals[:, i, :]
        y = era_vals[:, i, :]
        
        valid = ~np.isnan(x) & ~np.isnan(y)
        n_valid = np.sum(valid, axis=0)
        
        with np.errstate(invalid='ignore'):
            x_mean = np.nanmean(x, axis=0)
            y_mean = np.nanmean(y, axis=0)
            x_anom = x - x_mean
            y_anom = y - y_mean
            
            cov = np.nansum(x_anom * y_anom, axis=0)
            x_var = np.nansum(x_anom**2, axis=0)
            y_var = np.nansum(y_anom**2, axis=0)
            
            r = cov / np.sqrt(x_var * y_var)
            
            t_stat = r * np.sqrt((n_valid - 2) / (1 - r**2))
            pval = 2 * (1 - stats.t.cdf(np.abs(t_stat), n_valid - 2))
            
            corr[i, :] = r
            p_values[i, :] = pval

    print(f"[INFO] Finished {layer_name}")
    print(f"[STATS] Mean r: {np.nanmean(corr):.3f}")
    print(f"[STATS] Min r: {np.nanmin(corr):.3f}")
    print(f"[STATS] Max r: {np.nanmax(corr):.3f}")
    print(f"[STATS] Significant points (p<0.05): {np.sum(p_values < 0.05)}")

    return corr, p_values